# Healthcare semantic model — create DAX measures

Creates the 15 healthcare KPI measures from `resources/dax_measures.md` in an **existing** Direct Lake
semantic model, using [semantic-link-labs](https://github.com/microsoft/semantic-link-labs) over the
Tabular Object Model (TOM).

**Measures created**

| Folder | Measures | Home table |
|---|---|---|
| Volume | Total Encounters, Total Patients, ED Visits, Inpatient Admissions | `gold_encounter_summary` |
| Quality | Avg Length of Stay | `gold_encounter_summary` |
| Quality | Readmission Rate, Readmitted Patients, Avg Days to Readmission | `gold_readmissions` |
| Financial | Total Revenue, Total Billed, Collection Rate, Denial Rate, Revenue Lost to Denials | `gold_financial` |
| Population Health | Patients with Multimorbidity, Diabetes Prevalence | `gold_population_health` |

**Prerequisites**
- Run in a **Microsoft Fabric notebook**.
- Semantic model in a **Fabric / Premium-backed workspace** with **XMLA Endpoint = Read Write**
  (Workspace settings -> Premium).
- Notebook identity has **Contributor** or higher on the workspace.
- The four `gold_*` tables already exist in the model.

The notebook is **idempotent** — existing measures are updated in place, so it is safe to re-run.


## 1. Install / import dependencies

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import pandas as pd

import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

print("sempy-labs ready")

## 2. Target model

Set the semantic model name. Leave `WORKSPACE = None` to use the workspace this notebook is attached to.

`HOME_TABLE_OVERRIDE` lets you park **every** measure on one table (e.g. a dedicated `_Measures` table).
Leave it as `None` to store each measure on the table it queries.

In [ ]:
# --- EDIT ME -------------------------------------------------------------
DATASET = "Healthcare Analytics"   # semantic model (dataset) name or GUID
WORKSPACE = None                   # workspace name/GUID, or None for the current workspace
HOME_TABLE_OVERRIDE = None         # e.g. "_Measures" to put all measures on one table
# -------------------------------------------------------------------------

WORKSPACE = WORKSPACE or fabric.get_workspace_id()
print(f"Workspace: {WORKSPACE}")
print(f"Dataset:   {DATASET}")

## 3. Measure definitions

Transcribed from `dax_measures.md`. Rates use the `0.0%` format string and the `DIVIDE` third argument
(`0`) to guard against divide-by-zero.

In [ ]:
COUNT_FMT   = "#,0"
PCT_FMT     = "0.0%"
DAYS_FMT    = "0.0"
CURRENCY_FMT = "\$#,0;(\$#,0)"

MEASURES = [
    # ---------------- Core Volume ----------------
    {
        "table": "gold_encounter_summary",
        "name": "Total Encounters",
        "expression": """
COUNTROWS ( gold_encounter_summary )
""",
        "format_string": COUNT_FMT,
        "description": "Count of all encounter records.",
        "display_folder": "Volume",
    },
    {
        "table": "gold_encounter_summary",
        "name": "Total Patients",
        "expression": """
DISTINCTCOUNT ( gold_encounter_summary[patient_id] )
""",
        "format_string": COUNT_FMT,
        "description": "Distinct patients with at least one encounter.",
        "display_folder": "Volume",
    },
    {
        "table": "gold_encounter_summary",
        "name": "ED Visits",
        "expression": """
CALCULATE (
    COUNTROWS ( gold_encounter_summary ),
    gold_encounter_summary[encounter_type] = "ED"
)
""",
        "format_string": COUNT_FMT,
        "description": "Encounters with type 'ED'.",
        "display_folder": "Volume",
    },
    {
        "table": "gold_encounter_summary",
        "name": "Inpatient Admissions",
        "expression": """
CALCULATE (
    COUNTROWS ( gold_encounter_summary ),
    gold_encounter_summary[encounter_type] = "Inpatient"
)
""",
        "format_string": COUNT_FMT,
        "description": "Encounters with type 'Inpatient'.",
        "display_folder": "Volume",
    },

    # ---------------- Quality ----------------
    {
        "table": "gold_readmissions",
        "name": "Readmission Rate",
        "expression": """
DIVIDE (
    CALCULATE ( COUNTROWS ( gold_readmissions ), gold_readmissions[was_readmitted] = TRUE () ),
    COUNTROWS ( gold_readmissions ),
    0
)
""",
        "format_string": PCT_FMT,
        "description": "Share of index encounters followed by a readmission. National benchmark ~15%.",
        "display_folder": "Quality",
    },
    {
        "table": "gold_encounter_summary",
        "name": "Avg Length of Stay",
        "expression": """
CALCULATE (
    AVERAGE ( gold_encounter_summary[length_of_stay_days] ),
    gold_encounter_summary[encounter_type] = "Inpatient",
    gold_encounter_summary[length_of_stay_days] > 0
)
""",
        "format_string": DAYS_FMT,
        "description": "Average inpatient length of stay in days, excluding zero-day stays.",
        "display_folder": "Quality",
    },
    {
        "table": "gold_readmissions",
        "name": "Readmitted Patients",
        "expression": """
CALCULATE (
    COUNTROWS ( gold_readmissions ),
    gold_readmissions[was_readmitted] = TRUE ()
)
""",
        "format_string": COUNT_FMT,
        "description": "Count of readmission records flagged as readmitted.",
        "display_folder": "Quality",
    },
    {
        "table": "gold_readmissions",
        "name": "Avg Days to Readmission",
        "expression": """
CALCULATE (
    AVERAGE ( gold_readmissions[days_to_readmission] ),
    gold_readmissions[was_readmitted] = TRUE ()
)
""",
        "format_string": DAYS_FMT,
        "description": "Average days between discharge and readmission.",
        "display_folder": "Quality",
    },

    # ---------------- Financial ----------------
    {
        "table": "gold_financial",
        "name": "Total Revenue",
        "expression": """
SUM ( gold_financial[paid_amount] )
""",
        "format_string": CURRENCY_FMT,
        "description": "Total amount actually paid on claims.",
        "display_folder": "Financial",
    },
    {
        "table": "gold_financial",
        "name": "Total Billed",
        "expression": """
SUM ( gold_financial[claim_amount] )
""",
        "format_string": CURRENCY_FMT,
        "description": "Total amount billed across claims.",
        "display_folder": "Financial",
    },
    {
        "table": "gold_financial",
        "name": "Collection Rate",
        "expression": """
DIVIDE (
    SUM ( gold_financial[paid_amount] ),
    SUM ( gold_financial[claim_amount] ),
    0
)
""",
        "format_string": PCT_FMT,
        "description": "Paid amount as a share of billed amount.",
        "display_folder": "Financial",
    },
    {
        "table": "gold_financial",
        "name": "Denial Rate",
        "expression": """
DIVIDE (
    CALCULATE ( COUNTROWS ( gold_financial ), gold_financial[claim_status] = "Denied" ),
    COUNTROWS ( gold_financial ),
    0
)
""",
        "format_string": PCT_FMT,
        "description": "Share of claims with status 'Denied'.",
        "display_folder": "Financial",
    },
    {
        "table": "gold_financial",
        "name": "Revenue Lost to Denials",
        "expression": """
CALCULATE (
    SUM ( gold_financial[claim_amount] ),
    gold_financial[claim_status] = "Denied"
)
""",
        "format_string": CURRENCY_FMT,
        "description": "Billed amount on denied claims.",
        "display_folder": "Financial",
    },

    # ---------------- Population Health ----------------
    {
        "table": "gold_population_health",
        "name": "Patients with Multimorbidity",
        "expression": """
CALCULATE (
    DISTINCTCOUNT ( gold_population_health[patient_id] ),
    gold_population_health[chronic_condition_count] >= 3
)
""",
        "format_string": COUNT_FMT,
        "description": "Distinct patients with 3 or more chronic conditions.",
        "display_folder": "Population Health",
    },
    {
        "table": "gold_population_health",
        "name": "Diabetes Prevalence",
        "expression": """
DIVIDE (
    CALCULATE ( COUNTROWS ( gold_population_health ), gold_population_health[has_diabetes] = TRUE () ),
    COUNTROWS ( gold_population_health ),
    0
)
""",
        "format_string": PCT_FMT,
        "description": "Share of the population flagged as diabetic.",
        "display_folder": "Population Health",
    },
]

print(f"{len(MEASURES)} measures staged")
display(
    pd.DataFrame(
        [(m["display_folder"], m["name"], m["table"], m["format_string"]) for m in MEASURES],
        columns=["Folder", "Measure", "Home Table", "Format"],
    )
)

## 4. Pre-flight — confirm the tables and columns exist

Catches typos and schema drift *before* anything is written to the model.

In [ ]:
# The TOM metadata cache can be stale within a notebook session, which makes existing
# objects look absent. Always refresh it before reading model metadata.
fabric.refresh_tom_cache(workspace=WORKSPACE)

REQUIRED_COLUMNS = {
    "gold_encounter_summary": ["patient_id", "encounter_type", "length_of_stay_days"],
    "gold_readmissions": ["was_readmitted", "days_to_readmission"],
    "gold_financial": ["paid_amount", "claim_amount", "claim_status"],
    "gold_population_health": ["patient_id", "chronic_condition_count", "has_diabetes"],
}

tables_df = fabric.list_tables(dataset=DATASET, workspace=WORKSPACE)
columns_df = fabric.list_columns(dataset=DATASET, workspace=WORKSPACE)

model_tables = set(tables_df["Name"])
model_columns = {(r["Table Name"], r["Column Name"]) for _, r in columns_df.iterrows()}

issues = []
for table, cols in REQUIRED_COLUMNS.items():
    if table not in model_tables:
        issues.append(f"MISSING TABLE: {table}")
        continue
    for col in cols:
        if (table, col) not in model_columns:
            issues.append(f"MISSING COLUMN: {table}[{col}]")

if HOME_TABLE_OVERRIDE and HOME_TABLE_OVERRIDE not in model_tables:
    issues.append(f"MISSING TABLE (HOME_TABLE_OVERRIDE): {HOME_TABLE_OVERRIDE}")

if issues:
    print("Pre-flight found problems — fix these before running step 5:")
    for i in issues:
        print(f"  - {i}")
else:
    print(f"Pre-flight OK — all {len(REQUIRED_COLUMNS)} tables and their columns exist.")

## 5. Create / update the measures

**Why this commits per measure.** `connect_semantic_model(..., readonly=False)` only calls
`model.SaveChanges()` *once*, when the `with` block exits. `add_measure()` merely adds the object to the
in-memory model, so a naive loop reports "CREATED" for every measure *before anything has been sent to the
server* — and because `SaveChanges()` is a single transaction, one measure the server rejects takes the
whole batch down with it, while the printed results still claim success.

This cell calls `tom.model.SaveChanges()` after each measure, so:
- the reported status reflects what the **server** accepted, not what went into memory;
- one bad measure cannot roll back the other 14;
- a failure captures the real server error, and `UndoLocalChanges()` discards the rejected object so the
  session stays usable.

In [ ]:
results = []

with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    existing_tables = {t.Name for t in tom.model.Tables}
    existing_measures = {ms.Name for t in tom.model.Tables for ms in t.Measures}

    for m in MEASURES:
        name = m["name"]
        table = HOME_TABLE_OVERRIDE or m["table"]

        if table not in existing_tables:
            results.append((name, table, "SKIPPED", f"table '{table}' not found in model"))
            continue

        kwargs = {
            "expression": m["expression"].strip(),
            "format_string": m["format_string"],
            "description": m["description"],
            "display_folder": m["display_folder"],
        }

        intent = "UPDATED" if name in existing_measures else "CREATED"

        try:
            if intent == "UPDATED":
                tom.update_measure(measure_name=name, **kwargs)
            else:
                tom.add_measure(table_name=table, measure_name=name, **kwargs)

            # Commit this measure on its own so the status below is the server's verdict.
            tom.model.SaveChanges()
            results.append((name, table, intent, ""))

        except Exception as e:
            # Discard the rejected object, otherwise every later SaveChanges fails too.
            try:
                tom.model.UndoLocalChanges()
            except Exception:
                pass
            results.append((name, table, "FAILED", str(e).replace("\n", " ")[:400]))

df_results = pd.DataFrame(results, columns=["Measure", "Table", "Action", "Note"])
display(df_results)
print(df_results["Action"].value_counts().to_string())

failed = df_results[df_results["Action"].isin(["FAILED", "SKIPPED"])]
if not failed.empty:
    print("\n--- Problems ---")
    for _, r in failed.iterrows():
        print(f"[{r['Action']}] {r['Measure']}: {r['Note']}")

## 6. Recalculate the model

A `calculate` refresh re-evaluates the model so the new measures resolve immediately.
Harmless to skip — the cell swallows errors.

In [ ]:
try:
    labs.refresh_semantic_model(dataset=DATASET, workspace=WORKSPACE, refresh_type="calculate")
    print("Recalculated.")
except Exception as e:
    print(f"Recalculate skipped/failed (often safe to ignore): {e}")

## 7. Validate — authoritative reconciliation via DMV

`fabric.list_measures()` can read from the cached TOM metadata, so it may show a measure that was never
saved (or hide one that was). The `$system.tmschema_measures` DMV goes straight to the engine, so this is
the check to trust. Anything listed as `MISSING` here genuinely is not in the model.

In [ ]:
fabric.refresh_tom_cache(workspace=WORKSPACE)

dmv_measures = fabric.evaluate_dax(
    dataset=DATASET,
    workspace=WORKSPACE,
    dax_string="select [Name], [TableID], [DisplayFolder], [FormatString] from $system.tmschema_measures",
)
dmv_tables = fabric.evaluate_dax(
    dataset=DATASET,
    workspace=WORKSPACE,
    dax_string="select [ID] AS [TableID], [Name] AS [TableName] from $system.tmschema_tables",
)

in_model = dmv_measures.merge(dmv_tables, on="TableID", how="left")
display(
    in_model[in_model["Name"].isin({m["name"] for m in MEASURES})][
        ["TableName", "Name", "DisplayFolder", "FormatString"]
    ].sort_values(["DisplayFolder", "Name"])
)

expected = {m["name"] for m in MEASURES}
actual = set(in_model["Name"])
missing = sorted(expected - actual)

print(f"Expected {len(expected)} | found in model {len(expected & actual)} | missing {len(missing)}")
if missing:
    print("MISSING FROM MODEL:")
    for n in missing:
        print(f"  - {n}")
else:
    print("All measures confirmed present in the model.")

## 8. Validate — evaluate every measure with DAX

Returns the actual value of each KPI against the current data, so a bad column reference or
filter surfaces here rather than in a report visual.

In [ ]:
checks = []

for m in MEASURES:
    name = m["name"]
    try:
        value = fabric.evaluate_dax(
            dataset=DATASET,
            workspace=WORKSPACE,
            dax_string=f'EVALUATE ROW ( "Value", [{name}] )',
        ).iloc[0, 0]
        checks.append((m["display_folder"], name, "OK", value))
    except Exception as e:
        checks.append((m["display_folder"], name, "ERROR", str(e)[:300]))

df_checks = pd.DataFrame(checks, columns=["Folder", "Measure", "Status", "Value / Error"])
display(df_checks)

errors = df_checks[df_checks["Status"] == "ERROR"]
print("All measures evaluated successfully." if errors.empty else f"{len(errors)} measure(s) failed — see above.")

## 9. Diagnose & repair missing measures

Run this if step 7 reported anything as `MISSING`. It retries each missing measure **individually** and
prints the raw server error, which is what tells you *why* it was rejected — most often a column that does
not exist in the Direct Lake table, or a type mismatch such as `was_readmitted = TRUE()` when the column
landed as a string.

In [ ]:
fabric.refresh_tom_cache(workspace=WORKSPACE)

dmv_measures = fabric.evaluate_dax(
    dataset=DATASET,
    workspace=WORKSPACE,
    dax_string="select [Name] from $system.tmschema_measures",
)
missing = [m for m in MEASURES if m["name"] not in set(dmv_measures["Name"])]

if not missing:
    print("Nothing to repair — every measure is already in the model.")
else:
    print(f"Retrying {len(missing)} missing measure(s) individually...\n")
    repair = []

    with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
        existing_tables = {t.Name for t in tom.model.Tables}

        for m in missing:
            name = m["name"]
            table = HOME_TABLE_OVERRIDE or m["table"]

            if table not in existing_tables:
                repair.append((name, table, "SKIPPED", f"table '{table}' not in model"))
                continue

            try:
                tom.add_measure(
                    table_name=table,
                    measure_name=name,
                    expression=m["expression"].strip(),
                    format_string=m["format_string"],
                    description=m["description"],
                    display_folder=m["display_folder"],
                )
                tom.model.SaveChanges()
                repair.append((name, table, "CREATED", ""))
            except Exception as e:
                try:
                    tom.model.UndoLocalChanges()
                except Exception:
                    pass
                repair.append((name, table, "FAILED", str(e).replace("\n", " ")[:600]))

    df_repair = pd.DataFrame(repair, columns=["Measure", "Table", "Action", "Server error"])
    display(df_repair)

    for _, r in df_repair[df_repair["Action"] == "FAILED"].iterrows():
        print(f"\n--- {r['Measure']} ---\n{r['Server error']}")

## 10. Reference — conditional formatting thresholds

Apply these in the report (Visual -> Cell elements -> Background/Font color -> Rules), or use them to
build explicit status measures.

| Measure | Green | Yellow | Red |
|---|---|---|---|
| Readmission Rate | < 12% | 12–15% | > 15% |
| Avg Length of Stay (Inpatient) | < 4 days | 4–6 days | > 6 days |
| Collection Rate | > 85% | 70–85% | < 70% |
| Denial Rate | < 10% | 10–15% | > 15% |
| ED Volume (vs prior) | < 5% increase | 5–10% increase | > 10% increase |

The optional cell below creates matching `... Status` text measures ("Green"/"Yellow"/"Red") you can bind
directly to a KPI card or use for icon rules. Run it only if you want them.

In [ ]:
STATUS_MEASURES = [
    {
        "table": "gold_readmissions",
        "name": "Readmission Rate Status",
        "expression": """
VAR _v = [Readmission Rate]
RETURN
    SWITCH ( TRUE (), _v < 0.12, "Green", _v <= 0.15, "Yellow", "Red" )
""",
        "description": "Green < 12%, Yellow 12-15%, Red > 15%.",
        "display_folder": "Quality\\Status",
    },
    {
        "table": "gold_encounter_summary",
        "name": "Avg Length of Stay Status",
        "expression": """
VAR _v = [Avg Length of Stay]
RETURN
    SWITCH ( TRUE (), _v < 4, "Green", _v <= 6, "Yellow", "Red" )
""",
        "description": "Green < 4 days, Yellow 4-6 days, Red > 6 days.",
        "display_folder": "Quality\\Status",
    },
    {
        "table": "gold_financial",
        "name": "Collection Rate Status",
        "expression": """
VAR _v = [Collection Rate]
RETURN
    SWITCH ( TRUE (), _v > 0.85, "Green", _v >= 0.70, "Yellow", "Red" )
""",
        "description": "Green > 85%, Yellow 70-85%, Red < 70%.",
        "display_folder": "Financial\\Status",
    },
    {
        "table": "gold_financial",
        "name": "Denial Rate Status",
        "expression": """
VAR _v = [Denial Rate]
RETURN
    SWITCH ( TRUE (), _v < 0.10, "Green", _v <= 0.15, "Yellow", "Red" )
""",
        "description": "Green < 10%, Yellow 10-15%, Red > 15%.",
        "display_folder": "Financial\\Status",
    },
]

status_results = []

with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
    existing_tables = {t.Name for t in tom.model.Tables}
    existing_measures = {ms.Name for t in tom.model.Tables for ms in t.Measures}

    for m in STATUS_MEASURES:
        name = m["name"]
        table = HOME_TABLE_OVERRIDE or m["table"]

        if table not in existing_tables:
            status_results.append((name, table, "SKIPPED", f"table '{table}' not found"))
            continue

        kwargs = {
            "expression": m["expression"].strip(),
            "description": m["description"],
            "display_folder": m["display_folder"],
        }

        intent = "UPDATED" if name in existing_measures else "CREATED"

        try:
            if intent == "UPDATED":
                tom.update_measure(measure_name=name, **kwargs)
            else:
                tom.add_measure(table_name=table, measure_name=name, **kwargs)
            tom.model.SaveChanges()
            status_results.append((name, table, intent, ""))
        except Exception as e:
            try:
                tom.model.UndoLocalChanges()
            except Exception:
                pass
            status_results.append((name, table, "FAILED", str(e).replace("\n", " ")[:400]))

display(pd.DataFrame(status_results, columns=["Measure", "Table", "Action", "Note"]))

## 11. Optional — remove these measures

Rolls back everything this notebook created. Destructive: run deliberately.

In [ ]:
# TO_DELETE = {m["name"] for m in MEASURES} | {m["name"] for m in STATUS_MEASURES}
#
# with connect_semantic_model(dataset=DATASET, workspace=WORKSPACE, readonly=False) as tom:
#     for t in list(tom.model.Tables):
#         for ms in list(t.Measures):
#             if ms.Name in TO_DELETE:
#                 tom.remove_object(object=ms)
#                 print(f"Deleted [{ms.Name}] from '{t.Name}'")

## Troubleshooting

| Symptom | Fix |
|---|---|
| **Step 5 says `CREATED` but the measure is not in the model** | This is why step 5 commits per measure and step 7 reconciles against the DMV. If a measure is genuinely missing, run **step 9** — it retries that measure alone and prints the server's actual rejection reason. |
| **Everything reports `CREATED` even on a re-run** | Stale TOM metadata cache — existing measures were invisible to the in-memory model. The `fabric.refresh_tom_cache(workspace=...)` calls in steps 4, 7 and 9 fix this. Re-run from step 4. |
| `XMLA endpoint is not enabled` / read-only error | Workspace settings -> Premium -> **XMLA Endpoint = Read Write**. Needs Fabric/Premium capacity, not a Pro-only workspace. |
| `(401) Unauthorized` | Notebook identity lacks workspace write access — grant Contributor or higher. |
| Step 4 reports a missing table/column | Table and column names are case-sensitive and must match the Direct Lake model exactly. Compare against `fabric.list_columns(DATASET, workspace=WORKSPACE)`. |
| `was_readmitted` / `has_diabetes` comparison returns blank | The column is a string (`"true"`), not boolean. Change `= TRUE ()` to `= "true"` in the affected expression and re-run steps 5 and 8. |
| Measure shows `ERROR` in step 8 | Fix the `expression` in step 3, then re-run steps 5 and 8 — the upsert is idempotent. |
| New measures not visible in a report | Reopen/refresh the report; the field list is cached client-side. |
